In [1]:
from pathlib import Path
import xarray as xr
import os
import dask
import cfgrib

In [2]:
base_path = Path(os.getcwd())
repo_root = base_path.parents[2]

In [3]:
data_dir = Path(f"{repo_root}/DataGRIB")
files = sorted(list(data_dir.glob("*.grib")))

ds = xr.open_mfdataset(
    files, 
    engine='cfgrib', 
    combine='nested', 
    concat_dim='time',
    backend_kwargs={'indexpath': ''}
)

In [9]:
print(ds)

<xarray.Dataset> Size: 210MB
Dimensions:            (time: 6086, step: 4, latitude: 22, longitude: 49)
Coordinates:
  * time               (time) datetime64[ns] 49kB 2006-11-01 ... 2023-12-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 195kB dask.array<chunksize=(31, 4), meta=np.ndarray>
  * latitude           (latitude) float64 176B 56.42 56.3 56.17 ... 53.92 53.8
  * longitude          (longitude) float64 392B 20.9 21.02 21.15 ... 26.77 26.9
    number             int64 8B 0
    surface            float64 8B 0.0
    heightAboveGround  float64 8B 2.0
Data variables:
    tp                 (time, step, latitude, longitude) float32 105MB dask.array<chunksize=(31, 4, 22, 49), meta=np.ndarray>
    t2m                (time, step, latitude, longitude) float32 105MB dask.array<chunksize=(31, 4, 22, 49), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_ce

In [ ]:
ds.to_netcdf(f"{repo_root}/DataGRIB/lithuania_ecmwf_tp_2006_2023.nc")

In [ ]:
ds = xr.open_dataset(f"{repo_root}/DataGRIB/lithuania_ecmwf_tp_2006_2023.nc")

In [11]:
lats = [54.68, 54.89, 55.70] # Vilnius, Kaunas, Klaipėda
lons = [25.27, 23.90, 21.14]

subset_points = ds.sel(
    latitude=lats, 
    longitude=lons, 
    method='nearest'
)

print(subset_points)

<xarray.Dataset> Size: 2MB
Dimensions:            (time: 6086, step: 4, latitude: 3, longitude: 3)
Coordinates:
  * time               (time) datetime64[ns] 49kB 2006-11-01 ... 2023-12-31
  * step               (step) timedelta64[ns] 32B 06:00:00 ... 1 days 00:00:00
    valid_time         (time, step) datetime64[ns] 195kB dask.array<chunksize=(31, 4), meta=np.ndarray>
  * latitude           (latitude) float64 24B 54.67 54.92 55.67
  * longitude          (longitude) float64 24B 25.27 23.9 21.15
    number             int64 8B 0
    surface            float64 8B 0.0
    heightAboveGround  float64 8B 2.0
Data variables:
    tp                 (time, step, latitude, longitude) float32 876kB dask.array<chunksize=(31, 4, 3, 3), meta=np.ndarray>
    t2m                (time, step, latitude, longitude) float32 876kB dask.array<chunksize=(31, 4, 3, 3), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for M

In [ ]:
df = subset_points.to_dataframe()
df = df.reset_index()

df.to_csv(f"{repo_root}/DataGRIB/lithuania_ecmwf_tp_2mt_2006_2023.csv", index=False)